In [21]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import pandas as pd
import glob
import os
pd.set_option('display.max_columns', None)
# irene_df = pd.read_csv("../../data/evaluation_snippets/irene_v1/cc_hfpc_20min.Table.1.selections_IR.txt", sep="\t")
irene_df = pd.read_csv("../../data/evaluation_snippets/irene/cc_hfpc_5min_IR.txt", sep="\t")


In [24]:
# Add 'start_s' indicating the order within each SnippetFilename group (starting at 0)
irene_df['start_s'] = irene_df.groupby('SnippetFilename').cumcount()
irene_df["end_s"] = irene_df["start_s"] + 1

In [25]:
irene_df["BBPC"].value_counts()

BBPC
1    153
0    147
Name: count, dtype: int64

In [26]:
irene_df

,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),ECHO,HFPC,BBPC,Whistle,GROUNDTRUTH,DETAIL,Notes,SnippetFilename,start_s,end_s
0,1,Spectrogram 1,1,0.0,1.0,0.0,96000.0,0,1,0,1,hw,NaN,NaN,BSM_20170801_17291750.wav,0,1
1,2,Spectrogram 1,1,1.0,2.0,0.0,96000.0,0,1,1,1,bw,bmcm,NaN,BSM_20170801_17291750.wav,1,2
2,3,Spectrogram 1,1,2.0,3.0,0.0,96000.0,1,1,1,1,hbw,hmcm,NaN,BSM_20170801_17291750.wav,2,3
3,4,Spectrogram 1,1,3.0,4.0,0.0,96000.0,1,0,1,1,ebw,cm,NaN,BSM_20170801_16074520.wav,0,1
4,5,Spectrogram 1,1,4.0,5.0,0.0,96000.0,1,0,1,1,ebw,cm,NaN,BSM_20170801_16074520.wav,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,296,Spectrogram 1,1,295.0,296.0,0.0,96000.0,0,0,1,1,w,NaN,NaN,RDL_20200727_13003659.wav,1,2
296,297,Spectrogram 1,1,296.0,297.0,0.0,96000.0,0,0,0,1,w,NaN,NaN,RDL_20200727_13003659.wav,2,3
297,298,Spectrogram 1,1,297.0,298.0,0.0,96000.0,0,1,0,1,hw,hm,NaN,RDL_20200730_08495385.wav,0,1
298,299,Spectrogram 1,1,298.0,299.0,0.0,96000.0,1,1,0,1,ehw,hm,NaN,RDL_20200730_08495385.wav,1,2


In [27]:
import pandas as pd

# Extract site (first part before the first underscore)
irene_df["Site"] = irene_df["SnippetFilename"].str.split("_").str[0]



In [28]:
old_df = pd.read_csv("../../data/labels/Overlaps_1s.csv")


In [29]:
old_df[["HydrophoneModel", "HydrophoneSensitivity", "SnippetFilename", "Timestamp"]]

,HydrophoneModel,HydrophoneSensitivity,SnippetFilename,Timestamp
0,201359382,-172.7,BSM_20170724_13014300.wav,2017-07-24 13:01:43.00
1,201359382,-172.7,BSM_20170724_13014300.wav,2017-07-24 13:01:43.00
2,201359382,-172.7,BSM_20170724_13014300.wav,2017-07-24 13:01:43.00
3,201359382,-172.7,BSM_20170724_13062370.wav,2017-07-24 13:06:23.70
4,201359382,-172.7,BSM_20170724_13062370.wav,2017-07-24 13:06:23.70
...,...,...,...,...
10928,201359382,-172.7,CAC_20210714_09242300.wav,2021-07-14 09:24:23
10929,201359382,-172.7,CAC_20210714_09242800.wav,2021-07-14 09:24:28
10930,201359382,-172.7,CAC_20210714_09251600.wav,2021-07-14 09:25:16
10931,201359382,-172.7,CAC_20210714_09271800.wav,2021-07-14 09:27:18


In [30]:
# Get unique hydrophone info per snippet (avoid duplicates)
hydrophone_info = old_df[["SnippetFilename", "HydrophoneModel", "HydrophoneSensitivity", "Timestamp"]].drop_duplicates()

# Merge into irene_df
irene_df = irene_df.merge(hydrophone_info, on="SnippetFilename", how="left")

In [31]:
irene_df.rename(columns={"Timestamp": "snippet_start_time"}, inplace=True)
irene_df["snippet_start_time"] = pd.to_datetime(irene_df["snippet_start_time"])


In [33]:
irene_df


,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),ECHO,HFPC,BBPC,Whistle,GROUNDTRUTH,DETAIL,Notes,SnippetFilename,start_s,end_s,Site,HydrophoneModel,HydrophoneSensitivity,snippet_start_time
0,1,Spectrogram 1,1,0.0,1.0,0.0,96000.0,0,1,0,1,hw,NaN,NaN,BSM_20170801_17291750.wav,0,1,BSM,201359382,-172.7,2017-08-01 17:29:17.500
1,2,Spectrogram 1,1,1.0,2.0,0.0,96000.0,0,1,1,1,bw,bmcm,NaN,BSM_20170801_17291750.wav,1,2,BSM,201359382,-172.7,2017-08-01 17:29:17.500
2,3,Spectrogram 1,1,2.0,3.0,0.0,96000.0,1,1,1,1,hbw,hmcm,NaN,BSM_20170801_17291750.wav,2,3,BSM,201359382,-172.7,2017-08-01 17:29:17.500
3,4,Spectrogram 1,1,3.0,4.0,0.0,96000.0,1,0,1,1,ebw,cm,NaN,BSM_20170801_16074520.wav,0,1,BSM,201359382,-172.7,2017-08-01 16:07:45.200
4,5,Spectrogram 1,1,4.0,5.0,0.0,96000.0,1,0,1,1,ebw,cm,NaN,BSM_20170801_16074520.wav,1,2,BSM,201359382,-172.7,2017-08-01 16:07:45.200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,296,Spectrogram 1,1,295.0,296.0,0.0,96000.0,0,0,1,1,w,NaN,NaN,RDL_20200727_13003659.wav,1,2,RDL,5675,-176.5,2020-07-27 13:00:36.590
296,297,Spectrogram 1,1,296.0,297.0,0.0,96000.0,0,0,0,1,w,NaN,NaN,RDL_20200727_13003659.wav,2,3,RDL,5675,-176.5,2020-07-27 13:00:36.590
297,298,Spectrogram 1,1,297.0,298.0,0.0,96000.0,0,1,0,1,hw,hm,NaN,RDL_20200730_08495385.wav,0,1,RDL,5675,-176.5,2020-07-30 08:49:53.850
298,299,Spectrogram 1,1,298.0,299.0,0.0,96000.0,1,1,0,1,ehw,hm,NaN,RDL_20200730_08495385.wav,1,2,RDL,5675,-176.5,2020-07-30 08:49:53.850


In [34]:
irene_df["clip_start_time"] = irene_df["snippet_start_time"] + pd.to_timedelta(irene_df["start_s"], unit="s")
irene_df["clip_end_time"] = irene_df["clip_start_time"] + pd.to_timedelta(1, unit="s")

In [35]:
irene_df["clip_filename"] = irene_df["Site"] + "_" + irene_df["clip_start_time"].dt.strftime("%Y%m%d_%H%M%S%f").str[:-4] + ".wav"

# Check for duplicate clip_filenames
# Drop duplicate clip_filenames, keeping the first occurrence
irene_df = irene_df.drop_duplicates(subset="clip_filename", keep="first")

irene_df

,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),ECHO,HFPC,BBPC,Whistle,GROUNDTRUTH,DETAIL,Notes,SnippetFilename,start_s,end_s,Site,HydrophoneModel,HydrophoneSensitivity,snippet_start_time,clip_start_time,clip_end_time,clip_filename
0,1,Spectrogram 1,1,0.0,1.0,0.0,96000.0,0,1,0,1,hw,NaN,NaN,BSM_20170801_17291750.wav,0,1,BSM,201359382,-172.7,2017-08-01 17:29:17.500,2017-08-01 17:29:17.500,2017-08-01 17:29:18.500,BSM_20170801_17291750.wav
1,2,Spectrogram 1,1,1.0,2.0,0.0,96000.0,0,1,1,1,bw,bmcm,NaN,BSM_20170801_17291750.wav,1,2,BSM,201359382,-172.7,2017-08-01 17:29:17.500,2017-08-01 17:29:18.500,2017-08-01 17:29:19.500,BSM_20170801_17291850.wav
2,3,Spectrogram 1,1,2.0,3.0,0.0,96000.0,1,1,1,1,hbw,hmcm,NaN,BSM_20170801_17291750.wav,2,3,BSM,201359382,-172.7,2017-08-01 17:29:17.500,2017-08-01 17:29:19.500,2017-08-01 17:29:20.500,BSM_20170801_17291950.wav
3,4,Spectrogram 1,1,3.0,4.0,0.0,96000.0,1,0,1,1,ebw,cm,NaN,BSM_20170801_16074520.wav,0,1,BSM,201359382,-172.7,2017-08-01 16:07:45.200,2017-08-01 16:07:45.200,2017-08-01 16:07:46.200,BSM_20170801_16074520.wav
4,5,Spectrogram 1,1,4.0,5.0,0.0,96000.0,1,0,1,1,ebw,cm,NaN,BSM_20170801_16074520.wav,1,2,BSM,201359382,-172.7,2017-08-01 16:07:45.200,2017-08-01 16:07:46.200,2017-08-01 16:07:47.200,BSM_20170801_16074620.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,296,Spectrogram 1,1,295.0,296.0,0.0,96000.0,0,0,1,1,w,NaN,NaN,RDL_20200727_13003659.wav,1,2,RDL,5675,-176.5,2020-07-27 13:00:36.590,2020-07-27 13:00:37.590,2020-07-27 13:00:38.590,RDL_20200727_13003759.wav
296,297,Spectrogram 1,1,296.0,297.0,0.0,96000.0,0,0,0,1,w,NaN,NaN,RDL_20200727_13003659.wav,2,3,RDL,5675,-176.5,2020-07-27 13:00:36.590,2020-07-27 13:00:38.590,2020-07-27 13:00:39.590,RDL_20200727_13003859.wav
297,298,Spectrogram 1,1,297.0,298.0,0.0,96000.0,0,1,0,1,hw,hm,NaN,RDL_20200730_08495385.wav,0,1,RDL,5675,-176.5,2020-07-30 08:49:53.850,2020-07-30 08:49:53.850,2020-07-30 08:49:54.850,RDL_20200730_08495385.wav
298,299,Spectrogram 1,1,298.0,299.0,0.0,96000.0,1,1,0,1,ehw,hm,NaN,RDL_20200730_08495385.wav,1,2,RDL,5675,-176.5,2020-07-30 08:49:53.850,2020-07-30 08:49:54.850,2020-07-30 08:49:55.850,RDL_20200730_08495485.wav


In [36]:
def set_verif_flags(gt):
    if pd.isna(gt):
        return pd.Series([False, False, False, False])
    gt_str = str(gt)
    if 'a' in gt_str:
        return pd.Series([False, False, False, False])
    return pd.Series([
        'e' in gt_str,  # ECHO_verif
        'b' in gt_str,  # BBPC_verif
        'h' in gt_str,  # HFPC_verif
        'w' in gt_str   # Whislte_verif
    ])

irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = irene_df["GROUNDTRUTH"].apply(set_verif_flags)
# Convert ECHO, BBPC, HFPC, Whistle columns to 0/1 integers
irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]].astype(int)


C:\Users\Admin\AppData\Local\Temp\ipykernel_9432\3066683244.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = irene_df["GROUNDTRUTH"].apply(set_verif_flags)
C:\Users\Admin\AppData\Local\Temp\ipykernel_9432\3066683244.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]] = irene_df[["ECHO", "BBPC", "HFPC", "Whistle"]].astype(int)


## Clipping wav files

In [37]:
irene_df["BBPC"].value_counts()

BBPC
1    157
0    142
Name: count, dtype: int64

In [38]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

# Output directory
output_dir = "../../data/Verified_Dataset/clip_wavs/"
os.makedirs(output_dir, exist_ok=True)

snippets_dir = "../../data/Full_Dataset/Snippets_3s_wav/"


grouped = irene_df.groupby(["SnippetFilename"])

for (snippet_filename, ), group in tqdm(grouped, total=len(grouped)):
    # print(snippet_filename)
    source_path = os.path.join(snippets_dir, snippet_filename)
    try:
        # Load the entire snippet once
        y, sr = librosa.load(source_path, sr=None)
        
        # Extract all clips from this snippet
        for idx, row in group.iterrows():
            output_path = os.path.join(output_dir, row["clip_filename"])
            
            # Skip if already exists
            if os.path.exists(output_path):
                continue
            
            # Calculate sample indices
            start_sample = int(row["start_s"] * sr)
            end_sample = int(row["end_s"] * sr)
            
            # Extract and save the clip
            clip = y[start_sample:end_sample]
            sf.write(output_path, clip, sr)
            
    except Exception as e:
        print(f"Error processing {snippet_filename}: {e}")


100%|██████████| 100/100 [00:02<00:00, 44.91it/s]


In [39]:

irene_df = irene_df.drop(columns=["Selection", "View", "Channel","Begin Time (s)", "End Time (s)", "Low Freq (Hz)", "High Freq (Hz)"])


In [40]:
irene_df["Notes"].value_counts()
irene_df["Boat"] = irene_df["Notes"].str.contains("ship", case=False, na=False).astype(int)


In [42]:
irene_df["Boat"].value_counts()

Boat
0    231
1     68
Name: count, dtype: int64

In [43]:
irene_df.to_csv("../../data/Verified_Dataset/labels/labels_irene_5min.csv", index=False)

In [20]:
irene_df[irene_df["clip_filename"] == "BSM_20170801_16441900.wav"]

,ECHO,HFPC,BBPC,Whistle,GROUNDTRUTH,DETAIL,Notes,SnippetFilename,start_s,end_s,Site,HydrophoneModel,HydrophoneSensitivity,snippet_start_time,clip_start_time,clip_end_time,clip_filename,Boat
